# Практика 7: Spark

**Стек:**
- Jupyter (Python 3.8) → Spark 3.5.5 Standalone (Docker) → MinIO + PostgreSQL
- PostgreSQL: `postgres:5432`, база `shop`
- MinIO: `minio:9000`
- Пути в Spark — всегда **`s3a://`**

**UI:**
- Jupyter Notebook [`http://localhost:8888/lab`](http://localhost:8888/lab)
- Spark приложения [`http://localhost:4040`](http://localhost:4040)
- Master [`http://localhost:8080`](http://localhost:8080)
- Worker [`http://localhost:8081`](http://localhost:8081)
- MinIO Console [`http://localhost:9001`](http://localhost:9001)

**Полезные ссылки:**
- Модули, классы, функцим и методы PySpark [https://spark.apache.org/docs/latest/api/python/reference/](https://spark.apache.org/docs/latest/api/python/reference/)
- Конфигурация Spark [https://spark.apache.org/docs/latest/configuration.html](https://spark.apache.org/docs/latest/configuration.html)

> Если ноутбук выполняется **на Mac вне Docker**: `PG` → `localhost:5433`, `S3_ENDPOINT` → `http://localhost:9000`, убрать `.config("spark.driver.host", ...)`, `master` — `spark://localhost:7077`.

---

**Содержание:**
- Справка (JDBC, режимы записи, функции PySpark)
- 0. Константы, SparkSession (+ валидация окружения)
- 1. Чтение данных из PostgreSQL (JDBC) и запись в s3 — Задачи 1.1–1.3
- 2. Расчёт данных и запись в s3 — Задачи 2.1–2.4, Бонус-задача 2.5* (Delta)

> 💡 Заголовки сворачиваются в JupyterLab: клик по стрелке слева от заголовка
> (или Settings → Notebook → включить *Collapsible Headings*).
> Подэлементы задач (Требования / Проверка корректности результатов) — заголовки 4-го уровня, сворачиваются отдельно.

# Справка

_Справочный материал по JDBC, режимам записи и функциям PySpark. К разделам с задачами можно возвращаться по ссылкам из 💡-строк._


## JDBC (любая реляционная БД)

Конфиг для подключения к БД
```python
jdbc = dict(
    url="jdbc:postgresql://<host>:<port>/<database>",
    user="<user>",
    password="<password>",
    driver="org.postgresql.Driver",
)
```

Вся таблица:

```python
df = spark.read.format("jdbc").option("dbtable", "<schema>.<table>").options(**jdbc).load()
```

Подзапрос (фильтр выполняется на стороне БД — по сети идёт меньше данных):

```python
df = (
    spark.read.format("jdbc")
    .option("query", "SELECT <cols> FROM <table> WHERE <condition>")  # без скобок и ";"
    .options(**jdbc)
    .load()
)
```

Параллельное чтение большой таблицы (данные делятся по диапазонам ключа):

```python
df = (
    spark.read.format("jdbc")
    .option("dbtable", "<table>")
    .option("partitionColumn", "<numeric_column>")
    .option("lowerBound", "<min>")
    .option("upperBound", "<max>")
    .option("numPartitions", "<N>")
    .options(**jdbc)
    .load()
)
```

### Файлы

```python
df = spark.read.parquet("<path>")                                    # parquet
df = spark.read.format("delta").load("<path>")                       # delta
df = spark.read.format("csv").option("header", True).load("<path>")  # csv

# партиционированный датасет (каталоги вида col=value → значения станут колонками)
df = spark.read.format("parquet").load("<path>/")
```


## Полная перезапись (маленькие справочники)

```python
df.write.format("parquet").mode("overwrite").save("<path>")
```

### Партиционированная идемпотентная запись (факты)

> Требует `partitionOverwriteMode=dynamic` — перезапишутся **ТОЛЬКО**
> партиции, присутствующие в `df`; остальные не тронутся

```python
(
    df.withColumn("<partition_col>", F.to_date("<date_column>"))
    .write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("<partition_col>")
    .save("<path>")
)
```

То же, но конфиг локально на одну запись (не меняет сессию):

```python
(
    df.write
    .format("parquet")
    .mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")
    .partitionBy("<partition_col>")
    .save("<path>")
)
```

### Append (только вставки: события, логи)

```python
df.write.format("parquet").mode("append").save("<path>")
```

### Delta: условная перезапись с транзакционностью

```python
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "<partition_col> >= '<date_from>' AND <partition_col> < '<date_to>'")
    .save("<path>")
)
```

### Полезные опции

```python
df.write.option("compression", "snappy").save("<path>")    # кодек
df.coalesce(1).write...                                    # 1 файл вместо N (только для мелких данных!)
```


## Функции PySpark `pyspark.sql.functions`

Демо на `df_orders` (raw/orders из задач 1.2–1.3; чтобы ячейки выполнялись, сначала реши задачи раздела 1 или загрузи raw/orders).

Агрегаты → условные выражения → строки → даты → числа → window-функции → collect → тот же запрос на **Spark SQL** (один оптимизатор — один план).


In [ ]:
import pyspark.sql.functions as F

# 1) Агрегаты: count / sum / avg / min / max / countDistinct
df_orders.agg(
    F.count('*').alias('orders_cnt'),
    F.sum('total_amount').alias('revenue'),
    F.avg('total_amount').alias('avg_check'),
    F.min('total_amount').alias('min_check'),
    F.max('total_amount').alias('max_check'),
    F.countDistinct('customer_id').alias('clients')
).show()

# 2) Условные выражения: when/otherwise, coalesce (функция: первый не-null), isNotNull
labeled = (
    df_orders
    .withColumn('segment',
        F.when(F.col('total_amount') > 1000, 'vip')
         .when(F.col('total_amount') > 100, 'regular')
         .otherwise('small'))
    .withColumn('status_clean', F.coalesce(F.col('status'), F.lit('unknown')))
)
labeled.groupBy('segment').agg(
    F.count('*').alias('n'),
    F.sum('total_amount').alias('revenue'),
).orderBy('revenue').show()


In [ ]:
# 3) Строки и преобразование типов
df_orders.select(
    F.col('order_id'),
    F.upper('status').alias('status_up'),
    F.lower('status').alias('status_low'),
    F.trim('status').alias('status_trim'),
    F.concat(F.col('status'), F.lit('_'), F.col('order_id').cast('string')).alias('tag'),
    F.concat_ws('-', F.col('status'), F.lit('v1')).alias('tag_ws'),
    F.length('status').alias('len_status'),
    F.col('total_amount').cast('double').alias('amount_double'),
    F.col('total_amount').cast('string').alias('amount_str'),
).show(5)


In [ ]:
# 4) Даты и время
df_orders.select(
    F.to_date('order_created_at').alias('order_date'),
    F.year('order_created_at').alias('y'),
    F.month('order_created_at').alias('m'),
    F.dayofweek('order_created_at').alias('dow'),            # 1 = воскресенье
    F.date_format('order_created_at', 'yyyy-MM-dd HH').alias('ymdh'),
    F.datediff(F.current_date(), F.to_date('order_created_at')).alias('age_days'),
    F.date_add(F.to_date('order_created_at'), 7).alias('week_after'),
).orderBy(F.desc('order_date')).show(5)
# ⚠️ глобальный orderBy — это shuffle (Exchange в плане):
#    show(5) без сортировки или sortWithinPartitions — дешевле.


In [ ]:
# 5) Числовые функции
df_orders.select(
    F.col('total_amount'),
    F.round('total_amount', 0).alias('r0'),
    F.floor('total_amount').alias('fl'),
    F.ceil('total_amount').alias('ce'),
    F.abs('total_amount').alias('ab'),
    F.pow(F.col('total_amount'), 2).alias('sq'),
    F.greatest(F.col('total_amount'), F.lit(100)).alias('max_100'),
    F.least(F.col('total_amount'), F.lit(100)).alias('min_100'),
).show(5)


In [ ]:
from pyspark.sql.window import Window

# 6) Window-функции: row_number / rank / lag / lead
#    partitionBy + orderBy — это shuffle (Exchange в плане), несмотря на «оконную» природу
w_amt  = Window.partitionBy('customer_id').orderBy(F.desc('total_amount'))
w_time = Window.partitionBy('customer_id').orderBy('order_created_at')

ranked = df_orders.select(
    F.col('customer_id'),
    F.col('order_id'),
    F.col('total_amount'),
    F.row_number().over(w_amt).alias('rn'),
    F.rank().over(w_amt).alias('rk'),
    F.lag('total_amount').over(w_time).alias('prev_amount'),
    F.lead('total_amount').over(w_time).alias('next_amount'),
)
ranked.filter(F.col('rn') <= 2).show(10, truncate=False)


In [ ]:
# 7) Сборка значений: collect_list / collect_set — ВСЁ в одну строку!
#    Осторожно: при большом числе строк в группе переполните executor (аналог LISTAGG).
df_orders.groupBy('status').agg(
    F.collect_list('order_id').alias('order_ids'),
    F.collect_set('customer_id').alias('customers'),
).show(5, truncate=60)


# 0. Константы, SparkSession

In [ ]:
# Константы: PostgreSQL и S3 (MinIO)
# import os
# print(*os.environ.items(), sep="\n")

PG = dict(host='postgres', port=5432, dbname='shop', user='student', password='studentpass')
MINIO_ENDPOINT = 'http://minio:9000'
MINIO_ROOT_USER = 'minioadmin'
MINIO_ROOT_PASSWORD = 'minioadmin'
BUCKET = 'de-course'

In [ ]:
# Создать бакет, если его ещё нет (boto3)
import boto3
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ROOT_USER,
    aws_secret_access_key=MINIO_ROOT_PASSWORD,
    region_name='us-east-1',
    config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}),
)

existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f'Бакет {BUCKET} создан')
else:
    print(f'Бакет {BUCKET} уже существует')


In [ ]:
# Настройка SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("de-practice-7")
    .config("spark.driver.host", "jupyter")
    # jar-пакеты: в Docker-образе уже подключены через spark.jars в spark-defaults.conf
    # (/opt/spark/extra-jars: postgresql, hadoop-aws, aws-java-sdk, delta).
    # spark.jars.packages нужен, только если запускаешь ВНЕ Docker без предустановленных jar -
    # тогда раскомментируй, и Spark сам скачает зависимости из Maven.
    # .config(
    #     "spark.jars.packages",
    #     "org.postgresql:postgresql:42.7.3,org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-spark_2.12:3.3.0"
    # )
    # .config("spark.sql.adaptive.enabled", "false")                                                 # отключение AQE
    # .config("spark.sql.adaptive.autoBroadcastJoinThreshold", "-1")                                 # отключение авто-broadcast
    # .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")                     # подключение Delta Lake к сессии
    # .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")  # подменяет стандартный каталог на Delta — таблицы по путям становятся Delta-таблицами
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ROOT_USER)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_ROOT_PASSWORD)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| master:", spark.conf.get("spark.master"))
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"), "| auto-broadcast threshold:", spark.conf.get("spark.sql.adaptive.autoBroadcastJoinThreshold"))


In [ ]:
# Валидация окружения: быстро проверяем, что стек поднят и доступен
# (падение здесь - сигнал проверить docker-compose, а не задачи)
import socket

def check_tcp(host, port, name):
    try:
        socket.create_connection((host, port), timeout=3).close()
        print(f"OK    {name} ({host}:{port})")
        return True
    except Exception as e:
        print(f"FAIL  {name} ({host}:{port}): {e}")
        return False

check_tcp("spark-master", 7077, "Spark Master")
check_tcp("postgres", 5432, "PostgreSQL")
check_tcp("minio", 9000, "MinIO")

# MinIO: бакет доступен?
try:
    existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
    print(f"OK    MinIO бакеты: {sorted(existing)}")
except Exception as e:
    print(f"FAIL  MinIO list_buckets: {e}")

# Spark: версия, master и подключённые jar (из spark.jars в spark-defaults.conf)
print("Spark", spark.version, "| master:", spark.conf.get("spark.master"))
jars = spark.conf.get("spark.jars", "")
print("spark.jars:", jars if jars else "(не задан - см. комментарий в ячейке SparkSession)")

# PostgreSQL через Spark JDBC: читаем 1 строку
jdbc_check = dict(
    url=f"jdbc:postgresql://{PG['host']}:{PG['port']}/{PG['dbname']}",
    user=PG['user'],
    password=PG['password'],
    driver='org.postgresql.Driver',
)
try:
    n = spark.read.format("jdbc").option("query", "SELECT 1 AS ok").options(**jdbc_check).load().count()
    print("OK    PostgreSQL JDBC, SELECT 1:", n == 1)
except Exception as e:
    print("FAIL  PostgreSQL JDBC:", e)

# 1. Чтение данных из PostgreSQL (JDBC) и запись в s3

_Справочный материал перенесён в раздел «Справка» в начале ноутбука._


## Практика

Формат задач: в каждой задаче нужно написать `where`, `query` и определить колонку партиционирования.

Общее условие: данные в PostgreSQL могут меняться на глубину до 3 дней относительно дня загрузки (`LOAD_DATE`).

Запуск загрузки с локальной машины:
```bash
docker exec ubuntu-vm bash -c \
  'cd /home/student/workspace/store_hse/spark/raw/<name> && bash <name>.sh --load-date 2026-03-01'
```

In [ ]:
# Функции
import os
from datetime import datetime, timedelta
from pyspark.sql import functions as F

# Конфиг для подключения к Postgres
PG = dict(
    host='postgres',
    port=5432,
    dbname=os.environ.get('POSTGRES_DB'),
    user=os.environ.get('POSTGRES_USER'),
    password=os.environ.get('POSTGRES_PASSWORD'),
)

# Все пути к данным в S3 в одном месте
customers_path = f"s3a://{BUCKET}/raw/customers"
orders_path = f"s3a://{BUCKET}/raw/orders"
payments_path = f"s3a://{BUCKET}/raw/payments"
stg_orders_path = f"s3a://{BUCKET}/staging/stg_orders"
stg_payments_path = f"s3a://{BUCKET}/staging/stg_payments"
int_orders_with_payments_path = f"s3a://{BUCKET}/intermediate/int_orders_with_payments"
fct_orders_path = f"s3a://{BUCKET}/marts/fct_orders"

# ⚠️ В реальных загрузках LOAD_DATE - это дата запуска (datetime.now().date());
# здесь зафиксирована константой для воспроизводимости результатов
LOAD_DATE = "2026-03-05"

jdbc = dict(
    url=f"jdbc:postgresql://{PG['host']}:{PG['port']}/{PG['dbname']}",
    user=PG['user'],
    password=PG['password'],
    driver='org.postgresql.Driver',
)


def sink_has_data(path: str) -> bool:
    """True, если в приёмнике уже есть данные."""
    try:
        return spark.read.parquet(path).limit(1).count() > 0
    except Exception:
        return False


### Задача 1.1. Загрузка `raw/customers`

**Таблица-источник:** `public.customers` (PostgreSQL)

| Колонка | Смысл |
|---|---|
| `customer_id` | идентификатор клиента |
| `registered_at` | дата/время регистрации |
| `city` | город |
| `acquisition_channel` | канал привлечения |

#### Требования
1. Напиши `query` — снимок всех колонок справочника.
2. Определи, нужен ли `where` — учти, что справочник маленький и загружается полным снимком.
3. Определи колонку партиционирования или обоснуй, почему запись идёт без `partitionBy`.
4. Условие «данные меняются до 3 дней» для этой таблицы **не применяется** — объясни почему в комментарии к коду.
5. Ответь на вопрос: почему для этой загрузки не нужна проверка `sink_has_data`?

💡 _Справка: раздел «1. Инфо» → «Полная перезапись (маленькие справочники)»._

#### Проверка корректности результатов
- при повторном запуске данные в приёмнике не дублируются;
- результат идемпотентен при любом числе запусков.

In [ ]:
# Задача 1.1: raw/customers

# TODO(1): query - снимок всех колонок справочника
query = """
    SELECT ... FROM public.customers
"""

# TODO(2): where - нужна ли фильтрация для полного снимка? (обоснуй комментарием)
where = """

"""

df_customers = (
    spark.read.format("jdbc")
    .option("query", query)
    .options(**jdbc)
    .load()
)

# TODO(3): .partitionBy(...) или без partitionBy - обосновать
(
    df_customers.write
    .format("parquet")
    .mode("overwrite")
    .save(customers_path)
)

print(f"Сохранено: {customers_path}, строк: {df_customers.count()}")


In [ ]:
# Автопроверка: raw/customers

df = spark.read.parquet(customers_path)

dups = df.groupBy("customer_id").count().filter("count > 1").count()
assert dups == 0, f"дубликаты customer_id: {dups} - снимок не идемпотентен"

print(f"OK: {df.count()} строк, дубликатов customer_id нет")


### Задача 1.2. Загрузка `raw/orders`

**Таблица-источник:** `public.orders` (PostgreSQL)

| Колонка | Смысл |
|---|---|
| `order_id` | идентификатор заказа |
| `customer_id` | клиент |
| `order_created_at` | дата/время создания заказа |
| `status` | статус заказа |
| `total_amount` | сумма заказа |

#### Требования
1. Напиши `where` для инкрементной загрузки: если приёмник пуст — полная загрузка, если не пуст — окно `[LOAD_DATE - 3 дня; LOAD_DATE)` по дате создания заказа. Данные могут меняться в течение 3 дней, поэтому перечитываются все 3 последних дня.
2. Напиши `query` — все колонки + применённый `where` (фильтр должен выполняться на стороне PostgreSQL).
3. Определи колонку партиционирования: выведи из `order_created_at` колонку типа `date` через `F.to_date(...)`.
4. Обоснуй, почему нужен `partitionOverwriteMode=dynamic`: при overwrite перезаписываются только затронутые 3 партиции, остальные не трогаются.

💡 _Справка: раздел «1. Инфо» → «Партиционированная идемпотентная запись (факты)»; раздел «Функции PySpark» → п.4 «Даты и время»._

#### Проверка корректности результатов
- повторный запуск с той же `load-date` не создаёт дублей `order_id` внутри партиции;
- партиции вне окна `[LOAD_DATE-3; LOAD_DATE)` остаются нетронутыми;


<details><summary>Подсказка (задача 1.2)</summary>

Шаблон <code>where</code>:
<pre><code>WHERE order_created_at >= '&lt;start_date&gt;'::timestamp
  AND order_created_at &lt;  '&lt;load_date&gt;'::timestamp</code></pre>

Колонка партиции: <code>F.to_date("order_created_at")</code>.
</details>

In [ ]:
# Задача 1.2: raw/orders

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

if sink_has_data(orders_path):
    # TODO(1): where для инкремента [start_date .. load_date)
    where = f"""

    """
else:
    print("данных в приёмнике нет - полная загрузка таблицы")
    where = ""

# TODO(2): query - все колонки + where (фильтр на стороне PostgreSQL)
query = f"""
    SELECT ... FROM public.orders {where}
"""

df_orders = (
    spark.read.format("jdbc")
    .option("query", query)
    .options(**jdbc)
    .load()
)

# TODO(3): колонка партиционирования из order_created_at
df_orders = df_orders.withColumn(
    # ...
)

# TODO(4): запись с partitionBy (dynamic overwrite уже включён в сессии)
(
    df_orders.write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("TODO")
    .save(orders_path)
)

print(f"Сохранено: {orders_path}")


In [ ]:
# Автопроверка: raw/orders

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

df = spark.read.parquet(orders_path)

# интервал инкремента [start_date; load_date):
# партиции вне окна остаются от прошлых запусков, поэтому все проверки - только по окну
df_window = df.filter(
    (F.col("order_created_dt") >= start_date) & (F.col("order_created_dt") < load_date)
)

dups = df_window.groupBy("order_id", "order_created_dt").count().filter("count > 1").count()
assert dups == 0, f"дубликаты order_id в окне [{start_date}; {load_date}): {dups}"

n_window = df_window.count()
assert n_window > 0, f"пусто окно [{start_date}; {load_date})"

# вне окна данных текущего запуска быть не должно; строки там - данные прошлых запусков
n_outside = df.filter(
    (F.col("order_created_dt") < start_date) | (F.col("order_created_dt") >= load_date)
).count()

print(f"OK: всего {df.count()} строк (вне окна от прошлых запусков: {n_outside}), "
      f"окно [{start_date}; {load_date}) = {n_window} строк, дубликатов нет")


### Задача 1.3. Загрузка `raw/payments`

**Таблица-источник:** `public.payments` (PostgreSQL)

| Колонка | Смысл |
|---|---|
| `payment_id` | идентификатор платежа |
| `order_id` | заказ |
| `paid_at` | дата/время платежа, **может быть NULL** (платёж ещё не проведён) |
| `payment_status` | статус платежа |
| `amount` | сумма |

#### Требования
1. Напиши `where` для инкремента по `paid_at` с окном 3 дня. ⚠️ Ключевой подвох: платежи с `paid_at IS NULL` (pending) нужно **всегда** перечитывать независимо от окна — статус может смениться в любой момент. Проверь приоритет операторов: `AND` сильнее `OR`, скобки обязательны.
2. Напиши `query` — все колонки + применённый `where`.
3. Определи колонку партиционирования из `paid_at`. Объясни, в какую партицию попадут строки с `paid_at IS NULL` (`__HIVE_DEFAULT_PARTITION__`).
4. Обоснуй выбор `dynamic overwrite` по аналогии с orders.

💡 _Справка: раздел «1. Инфо» → «Партиционированная идемпотентная запись (факты)»; раздел «Функции PySpark» → п.4 «Даты и время»._

#### Проверка корректности результатов
- pending-платежи (`paid_at IS NULL`) попадают в приёмник при каждом инкрементном запуске;;
- партиции вне окна не перезаписываются.

<details><summary>Подсказка (задача 1.3)</summary>

Скобки обязательны, иначе <code>AND</code> «съест» соседнее условие из-за приоритета операторов:
<pre><code>WHERE (paid_at >= '&lt;start_date&gt;'::timestamp
   AND paid_at &lt;  '&lt;load_date&gt;'::timestamp)
   OR paid_at IS NULL</code></pre>

Колонка партиции: <code>F.to_date("paid_at")</code>; строки с <code>paid_at IS NULL</code> уйдут в <code>__HIVE_DEFAULT_PARTITION__</code>.
</details>

In [ ]:
# Задача 1.3: raw/payments

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

if sink_has_data(payments_path):
    # TODO(1): where для инкремента + платежи с paid_at IS NULL
    # подсказка: AND сильнее OR - расставь скобки
    where = f"""

    """
else:
    print("данных в приёмнике нет - полная загрузка таблицы")
    where = ""

# TODO(2): query - все колонки + where
query = f"""
    SELECT ... FROM public.payments {where}
"""

df_payments = (
    spark.read.format("jdbc")
    .option("query", query)
    .options(**jdbc)
    .load()
)

# TODO(3): колонка партиционирования из paid_at
df_payments = df_payments.withColumn(
    # ...
)

# TODO(4): запись с partitionBy
(
    df_payments.write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("TODO")
    .save(payments_path)
)

print(f"Сохранено: {payments_path}")


In [ ]:
# Автопроверка: raw/payments

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

df = spark.read.parquet(payments_path)

# интервал инкремента [start_date; load_date) + pending-платежи (paid_at IS NULL),
# которые догружаются при каждом запуске
df_window = df.filter(
    ((F.col("paid_at") >= start_date) & (F.col("paid_at") < load_date)) | F.col("paid_at").isNull()
)

dups = df_window.groupBy("payment_id").count().filter("count > 1").count()
assert dups == 0, f"дубликаты payment_id в окне [{start_date}; {load_date}) + pending: {dups}"

n_pending = df.filter(F.col("paid_at").isNull()).count()
assert n_pending > 0, "pending-платежи потеряны - проверь ветку OR paid_at IS NULL в where"

n_window = df_window.count()
assert n_window > 0, f"пусто окно [{start_date}; {load_date}) + pending"

print(f"OK: всего {df.count()} строк, окно [{start_date}; {load_date}) + pending = {n_window} строк, "
      f"pending: {n_pending}, дубликатов нет")


# 2. Расчёт данных и запись в s3

## Практика

Формат задач: дан SQL-запрос (как он выглядел бы в PostgreSQL) — нужно переписать его на PySpark с помощью `pyspark.sql.functions`.

Общие условия:
- источники — parquet-таблицы на S3, созданные в разделе «1. Практика» (пути `orders_path`, `payments_path` и т.д. заданы ранее);
- результаты сохраняются в parquet на S3 (`stg_orders_path`, `stg_payments_path`, `int_orders_with_payments_path`, `fct_orders_path`);
- при чтении источников применяй фильтр по партициям за окно `[LOAD_DATE - 3 дня; LOAD_DATE)` — интервал тот же, что в `where` при чтении из PostgreSQL (раздел 1);
- при записи используй идемпотентный паттерн: `mode("overwrite")` + `partitionBy(...)` (dynamic overwrite уже включён в сессии).

Подсказка по соответствию конструкций:

| PostgreSQL | PySpark |
|---|---|
| `select col as alias` | `F.col("col").alias("alias")` |
| `col::date` | `F.col("col").cast("date")` или `F.to_date("col")` |
| `expr = 'value'` (boolean) | `(F.col("expr") == "value")` |
| `and` / `not` | `&` / `~` (скобки обязательны) |
| `count(*) filter (where ...)` | `F.count(F.when(<условие>, 1))` |
| `sum(x) filter (where ...)` | `F.sum(F.when(<условие>, F.col("x")))` |
| `coalesce(x, 0)` | `F.coalesce(F.col("x"), F.lit(0))` |
| `min(x) filter (where ...)` | `F.min(F.when(<условие>, F.col("x")))` |
| `group by` | `.groupBy(...).agg(...)` |
| `a join b on ...` | `.join(df_b, on="key", how="left")` |
| `where not flag` | `.filter(~F.col("flag"))` |

### Задача 2.1. `stg_orders`

Дан SQL. Перепиши на PySpark: прочитай `orders_path`, примени все преобразования и сохрани результат в `stg_orders_path`.

```sql
with source as (
    select * from "shop"."public"."orders"
)

select
    order_id,
    customer_id,
    order_created_at                as ordered_at,
    order_created_at::date          as order_date,
    status                          as order_status,
    total_amount                    as order_amount,
    status = 'test'                 as is_test_order
from source
```

#### Требования
1. Приведи типы явно: `ordered_at` — timestamp, `order_date` — date.
2. Логическое выражение `status = 'test'` запиши как boolean-колонку через сравнение.
3. Сохрани в parquet с партиционированием по `order_date` (колонка уже выводится в select).

💡 _Справка: раздел «Функции PySpark» → п.3 «Строки и преобразование типов» (alias, cast)._

#### Проверка корректности результатов
- в результате ровно 7 колонок с именами из SQL;
- `order_date` имеет тип `date`, `is_test_order` — `boolean`;
- число строк в результате = числу строк в `orders_path` за то же окно;
- таблица партиционирована по `order_date`.

<details><summary>Подсказка (задача 2.1)</summary>

Нужны только <code>.select(...)</code> c <code>.alias()</code>, одним <code>.cast("date")</code> и одним булевым сравнением <code>(F.col("status") == "test")</code>.
</details>

In [ ]:
# Задача 2.1: stg_orders

# Окно инкремента - то же, что при чтении из БД в разделе 1: [LOAD_DATE - 3 дня; LOAD_DATE)
load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

# TODO(0): фильтр по партициям при чтении - интервал как в where задачи 1.2.
# Читай только нужные партиции (partition pruning), а не всю историю:
# df_orders = (
#     spark.read.format("parquet").load(orders_path)
#     .filter((F.col("order_created_dt") >= start_date) & (F.col("order_created_dt") < load_date))
# )
df_orders = spark.read.format("parquet").load(orders_path)

# TODO(1): перепиши SQL на PySpark
df_stg_orders = (
    df_orders
    # .select(...)
)

# TODO(2): сохрани результат в stg_orders_path
# таблица должна быть партиционирована по order_date (уже есть в select):
# df_stg_orders.write.format("parquet").mode("overwrite").partitionBy("order_date").save(...)

print(f"сохранено: {stg_orders_path}")


In [ ]:
# Автопроверка: stg_orders

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

df = spark.read.parquet(stg_orders_path)

assert len(df.columns) == 7, f"ожидалось 7 колонок, получено: {df.columns}"
dtypes = dict(df.dtypes)
assert dtypes["order_date"] == "date", f"order_date должен быть date, получен {dtypes['order_date']}"
assert dtypes["is_test_order"] == "boolean", f"is_test_order должен быть boolean, получен {dtypes['is_test_order']}"

# интервал [start_date; load_date): dynamic overwrite перезаписывает только партиции окна,
# поэтому приёмник сверяем с источником строго по окну, а не по всей таблице
df_window = df.filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))

# границы окна: min/max даты приёмника внутри интервала
bounds = df_window.agg(F.min("order_date").alias("mn"), F.max("order_date").alias("mx")).first()
assert bounds["mn"] >= start_date and bounds["mx"] < load_date, \
    f"данные окна выходят за границы [{start_date}; {load_date}): [{bounds['mn']}; {bounds['mx']}]"

# партиции: load_date - верхняя граница окна и в данные не входит,
# поэтому ориентируемся на start_date (первая партиция окна)
n_part = spark.read.parquet(f"{stg_orders_path}/order_date={start_date}").count()
assert n_part > 0, f"пуста партиция order_date={start_date}"

# число строк за окно = источнику за то же окно (partition pruning при чтении)
n_src = (
    spark.read.parquet(orders_path)
    .filter((F.col("order_created_dt") >= start_date) & (F.col("order_created_dt") < load_date))
    .count()
)
assert df_window.count() == n_src, f"число строк за окно [{start_date}; {load_date}) разошлось: {n_src} -> {df_window.count()}"

dups = df_window.groupBy("order_id", "order_date").count().filter("count > 1").count()
assert dups == 0, f"дубликаты (order_id, order_date) в окне: {dups}"

print(f"OK: {df_window.count()} строк за окно [{start_date}; {load_date}) - совпадает с источником, "
      f"схема корректна, дубликатов нет")


### Задача 2.2. `stg_payments`

Дан SQL. Перепиши на PySpark: прочитай `payments_path`, примени все преобразования и сохрани результат в `stg_payments_path`.

```sql
with source as (
    select * from "shop"."public"."payments"
)

select
    payment_id,
    order_id,
    paid_at,
    payment_status,
    amount,
    payment_status = 'success' and amount < 0      as is_refund
from source
```

#### Требования
1. Обрати внимание: в SQL выражение `payment_status = 'success' and amount < 0` — это одно boolean-условие, а не два разных поля.
2. В PySpark для `and` используется оператор `&`, операнды берутся в скобки.
3. Сохрани в parquet, партиционируй по дате платежа (`F.to_date("paid_at")`).

💡 _Справка: раздел «Функции PySpark» → п.2 «Условные выражения» (boolean через &)._

#### Проверка корректности результатов
- в результате 7 колонок (6 из SQL + `paid_dt` — колонка партиционирования), `is_refund` — `boolean`;
- таблица партиционирована по `paid_dt` (строки с `paid_at IS NULL` — в `__HIVE_DEFAULT_PARTITION__`);
- число строк = числу строк в `payments_path` за то же окно (включая строки с `paid_at IS NULL`).

<details><summary>Подсказка (задача 2.2)</summary>

Булево выражение - каждый операнд в скобках, объединение через <code>&amp;</code>:
<pre><code>((F.col("payment_status") == "success") &amp; (F.col("amount") &lt; 0))</code></pre>
</details>

In [ ]:
# Задача 2.2: stg_payments

# TODO(0): фильтр по партициям при чтении - интервал как в where задачи 1.3.
# ⚠️ не забудь pending-платежи (paid_dt IS NULL -> партиция __HIVE_DEFAULT_PARTITION__):
# df_payments = (
#     spark.read.format("parquet").load(payments_path)
#     .filter(
#         ((F.col("paid_dt") >= start_date) & (F.col("paid_dt") < load_date))
#         | F.col("paid_dt").isNull()
#     )
# )
df_payments = spark.read.format("parquet").load(payments_path)

# TODO(1): перепиши SQL на PySpark
df_stg_payments = (
    df_payments
    # .select(...)
)

# TODO(2): добавь колонку партиционирования (paid_dt из paid_at через F.to_date)
#          и сохрани в stg_payments_path с partitionBy("paid_dt"):
# df_stg_payments.write.format("parquet").mode("overwrite").partitionBy("paid_dt").save(...)

print(f"сохранено: {stg_payments_path}")


In [ ]:
# Автопроверка: stg_payments

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

df = spark.read.parquet(stg_payments_path)

assert len(df.columns) == 7, f"ожидалось 7 колонок, получено: {df.columns}"
dtypes = dict(df.dtypes)
assert dtypes["is_refund"] == "boolean", f"is_refund должен быть boolean, получен {dtypes['is_refund']}"
assert dtypes["paid_dt"] == "date", f"paid_dt должен быть date, получен {dtypes['paid_dt']}"

# интервал [start_date; load_date) + pending (paid_at IS NULL -> paid_dt IS NULL, default-партиция);
# сверяем приёмник с источником строго по окну, партиции вне окна не трогаем
df_window = df.filter(
    ((F.col("paid_dt") >= start_date) & (F.col("paid_dt") < load_date)) | F.col("paid_dt").isNull()
)

bounds = df_window.filter(F.col("paid_dt").isNotNull()).agg(
    F.min("paid_dt").alias("mn"), F.max("paid_dt").alias("mx")
).first()
assert bounds["mn"] >= start_date and bounds["mx"] < load_date, \
    f"данные окна выходят за границы [{start_date}; {load_date}): [{bounds['mn']}; {bounds['mx']}]"

# партиции: load_date - верхняя граница окна и в данные не входит,
# поэтому ориентируемся на start_date (первая партиция окна)
n_part = spark.read.parquet(f"{stg_payments_path}/paid_dt={start_date}").count()
assert n_part > 0, f"пуста партиция paid_dt={start_date}"

# число строк за окно (включая pending) = источнику за то же окно
n_src = (
    spark.read.parquet(payments_path)
    .filter(((F.col("paid_dt") >= start_date) & (F.col("paid_dt") < load_date)) | F.col("paid_dt").isNull())
    .count()
)
assert df_window.count() == n_src, f"число строк за окно [{start_date}; {load_date}) разошлось: {n_src} -> {df_window.count()}"

dups = df_window.groupBy("payment_id").count().filter("count > 1").count()
assert dups == 0, f"дубликаты payment_id в окне + pending: {dups}"

print(f"OK: {df_window.count()} строк за окно [{start_date}; {load_date}) + pending - совпадает с источником, дубликатов нет")


### Задача 2.3. `int_orders_with_payments`

Самая сложная задача. Дан SQL с CTE. Перепиши на PySpark: прочитай `stg_orders_path` и `stg_payments_path`, воспроизведи оба CTE и сохрани результат в `int_orders_with_payments_path`.

```sql
with orders as (
    select * from "shop"."analytics_staging"."stg_orders"
),

payments as (
    select * from "shop"."analytics_staging"."stg_payments"
),

order_payments as (
    select
        order_id,
        count(*) filter (where payment_status = 'success' and not is_refund)                 as successful_payments_count,
        coalesce(sum(amount) filter (where payment_status = 'success' and not is_refund), 0) as paid_amount,
        coalesce(-sum(amount) filter (where is_refund), 0)                                   as refunded_amount,
        min(paid_at) filter (where payment_status = 'success' and not is_refund)             as first_paid_at,
        count(*) filter (where payment_status = 'success')                                   as success_operations_count,
        count(*) filter (where payment_status = 'failed')                                    as failed_operations_count,
        count(*) filter (where payment_status = 'pending')                                   as pending_operations_count
    from payments
    group by order_id
)

select
    o.order_id,
    o.customer_id,
    o.ordered_at,
    o.order_date,
    o.order_status,
    o.order_amount,
    o.is_test_order,
    coalesce(p.successful_payments_count, 0)                        as successful_payments_count,
    coalesce(p.paid_amount, 0)                                      as paid_amount,
    coalesce(p.refunded_amount, 0)                                  as refunded_amount,
    coalesce(p.paid_amount, 0) - coalesce(p.refunded_amount, 0)     as net_paid_amount,
    p.first_paid_at,
    o.order_status = 'paid'
        and coalesce(p.successful_payments_count, 0) > 0            as is_confirmed,
    coalesce(p.success_operations_count, 0)                         as success_operations_count,
    coalesce(p.failed_operations_count, 0)                          as failed_operations_count,
    coalesce(p.pending_operations_count, 0)                         as pending_operations_count
from orders as o
left join order_payments as p
    on p.order_id = o.order_id
```

#### Требования
1. CTE `order_payments` — это агрегация платежей до уровня заказа: `groupBy("order_id").agg(...)`.
2. `count(*) filter (where ...)` → `F.count(F.when(<условие>, 1))`; для сумм и `min` — аналогично через `F.when`.
3. ⚠️ Внимание к `refunded_amount`: в SQL стоит `-sum(...)` — минус применяется к сумме **после** агрегации, а не к каждой строке. Оба варианта дадут одинаковый результат только при аккуратной записи — проверь себя.
4. Заказы без платежей должны остаться в результате (LEFT JOIN) — счётчики и суммы заменяются нулями через `coalesce`, а `first_paid_at` остаётся `NULL`.
5. `is_confirmed` — составное boolean-условие по двум полям после join.
6. Сохрани в parquet с партиционированием по `order_date`.

💡 _Справка: раздел «Функции PySpark» → п.1 «Агрегаты» и п.2 «Условные выражения» (count/sum через when)._

#### Проверка корректности результатов
- в результате 16 колонок;
- для заказа без платежей: `successful_payments_count = 0`, `paid_amount = 0`, `net_paid_amount = 0`, `first_paid_at IS NULL`, `is_confirmed = false`;
- число строк = числу строк в `stg_orders` (никакие заказы не потеряны и не задублированы);

<details><summary>Подсказка (задача 2.3)</summary>

Агрегаты с условием: <code>F.count(F.when(&lt;условие&gt;, 1))</code>; суммы и <code>min</code> - так же через <code>F.when</code>.
Join: <code>.join(df_order_payments, on="order_id", how="left")</code>.
Нули вместо NULL: <code>F.coalesce(F.col("..."), F.lit(0))</code>.
</details>

In [ ]:
# Задача 2.3: int_orders_with_payments

# TODO(0): фильтр по партициям/датам при чтении - окно то же, что при чтении из БД.
# stg_orders - по колонке партиционирования из задачи 2.1 (order_date):
#   .filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))
# stg_payments - по колонке партиционирования из задачи 2.2 (paid_dt),
# pending (IS NULL) должен остаться:
#   .filter(((F.col("paid_dt") >= start_date) & (F.col("paid_dt") < load_date)) | F.col("paid_dt").isNull())
df_stg_orders = spark.read.format("parquet").load(stg_orders_path)
df_stg_payments = spark.read.format("parquet").load(stg_payments_path)

# TODO(1): CTE order_payments -> агрегация платежей до уровня заказа
# df_order_payments = (
#     df_stg_payments
#     .groupBy("order_id")
#     .agg(...)
# )

# TODO(2): left join + финальный select с coalesce и is_confirmed
# df_int_orders_with_payments = (
#     df_stg_orders
#     .join(df_order_payments, on="order_id", how="left")
#     .select(...)
# )

# TODO(3): сохрани в int_orders_with_payments_path с partitionBy("order_date")
# df_int_orders_with_payments.write.format("parquet").mode("overwrite").partitionBy("order_date").save(int_orders_with_payments_path)

print(f"сохранено: {int_orders_with_payments_path}")


In [ ]:
# Автопроверка: int_orders_with_payments

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

df = spark.read.parquet(int_orders_with_payments_path)

assert len(df.columns) == 16, f"ожидалось 16 колонок, получено {len(df.columns)}"

# интервал [start_date; load_date): приёмник содержит и данные прошлых запусков,
# поэтому сравниваем со stg_orders строго по окну
df_window = df.filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))

# границы окна: min/max даты приёмника внутри интервала
bounds = df_window.agg(F.min("order_date").alias("mn"), F.max("order_date").alias("mx")).first()
assert bounds["mn"] >= start_date and bounds["mx"] < load_date, \
    f"данные окна выходят за границы [{start_date}; {load_date}): [{bounds['mn']}; {bounds['mx']}]"

# ни один заказ окна не потерян и не задублирован
n_src = (
    spark.read.parquet(stg_orders_path)
    .filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))
    .count()
)
assert df_window.count() == n_src, f"число строк за окно [{start_date}; {load_date}) разошлось: {n_src} -> {df_window.count()}"

dups = df_window.groupBy("order_id", "order_date").count().filter("count > 1").count()
assert dups == 0, f"дубликаты (order_id, order_date) в окне: {dups}"

# заказ без платежей: суммы = 0, first_paid_at = NULL, is_confirmed = false
row = df_window.filter(F.col("successful_payments_count") == 0).first()
if row is not None:
    assert row["paid_amount"] == 0 and row["net_paid_amount"] == 0, "суммы должны быть 0"
    assert row["first_paid_at"] is None, "first_paid_at должен быть NULL"
    assert row["is_confirmed"] == False, "is_confirmed должен быть false"

print(f"OK: {df_window.count()} строк за окно [{start_date}; {load_date}) - совпадает со stg_orders, дубликатов нет")


### Задача 2.4. `fct_orders`

Дан SQL. Перепиши на PySpark: прочитай `int_orders_with_payments_path`, примени фильтр и проекцию, сохрани результат в `fct_orders_path`.

```sql
select
    order_id,
    customer_id,
    order_date,
    ordered_at,
    order_status,
    order_amount,
    paid_amount,
    refunded_amount,
    net_paid_amount,
    successful_payments_count,
    first_paid_at,
    is_confirmed
from "shop"."analytics_intermediate"."int_orders_with_payments"
where not is_test_order
```

#### Требования
1. `where not is_test_order` → `.filter(~F.col("is_test_order"))`.
2. Порядок действий: фильтр можно применить до или после `select` — результат одинаков, но фильтр до `select` дешевле (меньше данных в проекции).
3. Сохрани в parquet с партиционированием по `order_date`.

💡 _Справка: раздел «Функции PySpark» → п.2 «Условные выражения» (фильтр через ~)._

#### Проверка корректности результатов
- в результате 12 колонок;
- строк с `is_test_order = true` в результате нет;
- число строк = числу строк в `int_orders_with_payments` минус число тестовых заказов;

<details><summary>Подсказка (задача 2.4)</summary>

Фильтр: <code>.filter(~F.col("is_test_order"))</code> - применяй его <b>до</b> <code>.select(...)</code>, чтобы не тянуть лишние колонки.
</details>

In [ ]:
# Задача 2.4: fct_orders

# TODO(0): фильтр по партициям при чтении - окно как при чтении из БД:
# df_int_orders_with_payments = (
#     spark.read.format("parquet").load(int_orders_with_payments_path)
#     .filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))
# )
df_int_orders_with_payments = spark.read.format("parquet").load(int_orders_with_payments_path)

# TODO(1): перепиши SQL на PySpark (filter + select)
# df_fct_orders = (
#     df_int_orders_with_payments
#     ...
# )

# TODO(2): сохрани в fct_orders_path с partitionBy("order_date")
# df_fct_orders.write.format("parquet").mode("overwrite").partitionBy("order_date").save(fct_orders_path)

print(f"сохранено: {fct_orders_path}")


In [ ]:
# Автопроверка: fct_orders

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

df = spark.read.parquet(fct_orders_path)

assert len(df.columns) == 12, f"ожидалось 12 колонок, получено {len(df.columns)}"

# интервал [start_date; load_date): сверяем приёмник с int только по окну
df_window = df.filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))

# границы окна: min/max даты приёмника внутри интервала
bounds = df_window.agg(F.min("order_date").alias("mn"), F.max("order_date").alias("mx")).first()
assert bounds["mn"] >= start_date and bounds["mx"] < load_date, \
    f"данные окна выходят за границы [{start_date}; {load_date}): [{bounds['mn']}; {bounds['mx']}]"

# строк за окно = int за окно минус тестовые
df_int = (
    spark.read.parquet(int_orders_with_payments_path)
    .filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))
)
n_expected = df_int.filter(F.col("is_test_order") == False).count()
assert df_window.count() == n_expected, f"ожидалось {n_expected} строк за окно, получено {df_window.count()}"

dups = df_window.groupBy("order_id", "order_date").count().filter("count > 1").count()
assert dups == 0, f"дубликаты (order_id, order_date) в окне: {dups}"

print(f"OK: {df_window.count()} строк за окно [{start_date}; {load_date}) - "
      f"int_orders_with_payments без тестовых заказов, дубликатов нет")


### Бонус-задача 2.5*. `fct_orders` на Delta Lake

_Необязательная задача повышенной сложности. Delta-библиотеки уже есть в Docker-образе (`delta-spark_2.12-3.3.0` в `/opt/spark/extra-jars`)._

#### Требования
1. Включи Delta в SparkSession: раскомментируй `spark.sql.extensions` и `spark.sql.catalog.spark_catalog` в ячейке «Настройка SparkSession» и перезапусти kernel.
2. Перепиши запись `fct_orders` в формат **delta** с идемпотентной перезаписью только нужного окна дат через `replaceWhere`.
3. Сравни с parquet-вариантом (задача 2.4): объём кода, поведение при повторном запуске, атомарность.

💡 _Справка: раздел «1. Инфо» → «Delta: условная перезапись с транзакционностью»._

In [ ]:
# Бонус-задача 2.5*: fct_orders на Delta

FCT_DELTA_PATH = f"s3a://{BUCKET}/marts/fct_orders_delta"

# TODO(1): прочитай int_orders_with_payments_path с фильтром по партициям за окно
#          [start_date; load_date), примени фильтр ~is_test_order и проекцию (как в задаче 2.4)

# TODO(2): запиши в формате delta с replaceWhere по окну order_date
# (
#     df_fct_orders.write
#     .format("delta")
#     .mode("overwrite")
#     .option("replaceWhere", f"order_date >= '{start_date}' AND order_date < '{load_date}'")
#     .save(FCT_DELTA_PATH)
# )

# TODO(3): прочитай delta-таблицу и убедись, что данные за пределами окна не тронуты
#          (сравни количество строк до и после повторного запуска)


In [ ]:
# Автопроверка: fct_orders

load_date = datetime.strptime(LOAD_DATE, "%Y-%m-%d").date()
start_date = load_date - timedelta(days=3)

df = spark.read.parquet(fct_orders_path)

assert len(df.columns) == 12, f"ожидалось 12 колонок, получено {len(df.columns)}"

# интервал [start_date; load_date): сверяем приёмник с int только по окну
df_window = df.filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))

# границы окна: min/max даты приёмника внутри интервала
bounds = df_window.agg(F.min("order_date").alias("mn"), F.max("order_date").alias("mx")).first()
assert bounds["mn"] >= start_date and bounds["mx"] < load_date, \
    f"данные окна выходят за границы [{start_date}; {load_date}): [{bounds['mn']}; {bounds['mx']}]"

# строк за окно = int за окно минус тестовые
df_int = (
    spark.read.parquet(int_orders_with_payments_path)
    .filter((F.col("order_date") >= start_date) & (F.col("order_date") < load_date))
)
n_expected = df_int.filter(F.col("is_test_order") == False).count()
assert df_window.count() == n_expected, f"ожидалось {n_expected} строк за окно, получено {df_window.count()}"

dups = df_window.groupBy("order_id", "order_date").count().filter("count > 1").count()
assert dups == 0, f"дубликаты (order_id, order_date) в окне: {dups}"

print(f"OK: {df_window.count()} строк за окно [{start_date}; {load_date}) - "
      f"int_orders_with_payments без тестовых заказов, дубликатов нет")
